# 03 — Visualizaciones

Genera las 4 gráficas del tablero a partir de los CSV de análisis.

**Datos:** carpeta `ENOE_analisis/` en Drive  
**Salida:** archivos HTML interactivos en `dashboard/`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

RUTA_ANALISIS = '/content/drive/MyDrive/ENOE_analisis'
RUTA_DASH     = '/content/drive/MyDrive/dashboard'
os.makedirs(RUTA_DASH, exist_ok=True)

# Paleta fija para que todas las gráficas sean consistentes
COLOR_HOMBRE = '#378ADD'
COLOR_MUJER  = '#D4537E'
COLOR_MAPA   = {'Hombre': COLOR_HOMBRE, 'Mujer': COLOR_MUJER}

In [3]:
# Cargar los 4 datasets
sn  = pd.read_csv(f'{RUTA_ANALISIS}/serie_nacional.csv',           encoding='utf-8-sig')
bs  = pd.read_csv(f'{RUTA_ANALISIS}/brecha_sector_anio.csv',        encoding='utf-8-sig')
be  = pd.read_csv(f'{RUTA_ANALISIS}/brecha_educ_sector_2025.csv',   encoding='utf-8-sig')
inf = pd.read_csv(f'{RUTA_ANALISIS}/informalidad_contexto_2025.csv',encoding='utf-8-sig')

# Brecha nacional anual para anotaciones
pivot_sn = sn.pivot(index='anio', columns='sexo', values='mediana_ing_hora').reset_index()
pivot_sn['brecha_pct'] = ((pivot_sn['Hombre'] - pivot_sn['Mujer']) / pivot_sn['Hombre'] * 100).round(1)

print('Datos cargados correctamente.')
print(f'  Serie nacional: {sn.shape}  |  Brecha sectorial: {bs.shape}')
print(f'  Brecha educativa: {be.shape}  |  Informalidad: {inf.shape}')

Datos cargados correctamente.
  Serie nacional: (42, 3)  |  Brecha sectorial: (231, 6)
  Brecha educativa: (55, 5)  |  Informalidad: (22, 3)


---
## G1 — Tendencia nacional 2005–2025

Muestra la evolución del ingreso mediano por hora de hombres y mujeres.
La brecha bajó de 7% en 2005 a 3.3% en 2025, pero no ha desaparecido.

In [4]:
fig1 = px.line(
    sn,
    x='anio', y='mediana_ing_hora', color='sexo',
    color_discrete_map=COLOR_MAPA,
    markers=True,
    labels={'anio': 'Año', 'mediana_ing_hora': 'Ingreso mediano por hora (pesos)', 'sexo': ''},
    title='Ingreso mediano por hora trabajada · México 2005–2025',
)

# Área entre las dos líneas para visualizar la brecha
hombre = sn[sn['sexo']=='Hombre'].sort_values('anio')
mujer  = sn[sn['sexo']=='Mujer'].sort_values('anio')

fig1.add_trace(go.Scatter(
    x=list(hombre['anio']) + list(hombre['anio'])[::-1],
    y=list(hombre['mediana_ing_hora']) + list(mujer['mediana_ing_hora'])[::-1],
    fill='toself',
    fillcolor='rgba(212,83,126,0.08)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=False,
    hoverinfo='skip',
))

# Anotación COVID-2020
fig1.add_vline(
    x=2020, line_dash='dot', line_color='gray', line_width=1.5,
    annotation_text='COVID-19', annotation_position='top left',
    annotation_font=dict(size=11, color='gray'),
)

# Anotación de brecha en el último año
ultimo = pivot_sn[pivot_sn['anio']==2025].iloc[0]
fig1.add_annotation(
    x=2025,
    y=(ultimo['Hombre'] + ultimo['Mujer']) / 2,
    text=f"Brecha: {ultimo['brecha_pct']}%",
    showarrow=True, arrowhead=2, ax=60, ay=0,
    font=dict(size=11, color=COLOR_MUJER),
    arrowcolor=COLOR_MUJER,
)

fig1.update_layout(
    template='plotly_white',
    legend=dict(orientation='h', y=-0.15),
    hovermode='x unified',
    xaxis=dict(dtick=2),
    font=dict(family='Arial, sans-serif', size=12),
    margin=dict(t=60, b=60, l=60, r=40),
)

fig1.write_html(f'{RUTA_DASH}/g1_serie_nacional.html')
fig1.show()
print('G1 guardada.')

G1 guardada.


---
## G2 — Brecha salarial por sector (2025)

Brecha porcentual: positivo = hombres ganan más. Negativo = mujeres ganan más.
Los sectores con brecha negativa (Industria extractiva, Construcción, Transportes, Gobierno)
tienen muy pocas mujeres — las que trabajan ahí tienden a ocupar puestos mejor pagados.
El hallazgo importante está en los sectores con más presencia femenina: Manufactura, Comercio y Restaurantes.

In [5]:
bs25 = bs[bs['anio'] == 2025].sort_values('brecha_pct', ascending=True).copy()

# Color por signo de brecha
bs25['color_barra'] = bs25['brecha_pct'].apply(
    lambda x: 'Hombres ganan más' if x > 0 else 'Mujeres ganan más'
)

fig2 = px.bar(
    bs25,
    x='brecha_pct', y='sector',
    color='color_barra',
    color_discrete_map={
        'Hombres ganan más': COLOR_MUJER,    # rojo/rosa = desventaja para mujeres
        'Mujeres ganan más': COLOR_HOMBRE,   # azul = ventaja para mujeres
    },
    orientation='h',
    text='brecha_pct',
    labels={'brecha_pct': 'Brecha salarial (%)', 'sector': '', 'color_barra': ''},
    title='Brecha salarial por sector económico · 2025<br><sup>% de diferencia en ingreso mediano por hora (positivo = hombres ganan más)</sup>',
)

fig2.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    textfont=dict(size=11),
)

# Línea vertical en 0
fig2.add_vline(x=0, line_color='black', line_width=1)

# Anotación de contexto sobre manufactura (mayor brecha con mayor presencia femenina)
fig2.add_annotation(
    x=14.1, y='Manufactura',
    text=' ← sector con alta presencia femenina',
    showarrow=False, xanchor='left',
    font=dict(size=10, color='gray'),
)

fig2.update_layout(
    template='plotly_white',
    legend=dict(orientation='h', y=-0.12),
    xaxis=dict(zeroline=False, range=[-35, 25]),
    font=dict(family='Arial, sans-serif', size=12),
    margin=dict(t=80, b=60, l=230, r=60),
    height=480,
)

fig2.write_html(f'{RUTA_DASH}/g2_brecha_sector.html')
fig2.show()
print('G2 guardada.')

G2 guardada.


---
## G3 — Heatmap: brecha por sector y nivel educativo (2025)

Responde la pregunta central: ¿desaparece la brecha cuando hombres y mujeres
tienen el mismo nivel de estudios?

Rojo = hombres ganan más. Azul = mujeres ganan más. Blanco = sin diferencia.
La manufactura muestra brecha persistente en todos los niveles educativos.

In [6]:
# Filtrar 'No especificado' por tener pocos casos y ser ruidoso
be_clean = be[be['nivel_educ'] != 'No especificado'].copy()

# Ordenar niveles educativos de menor a mayor
orden_educ = ['Sin instrucción', 'Básica', 'Media superior', 'Superior']
be_clean['nivel_educ'] = pd.Categorical(be_clean['nivel_educ'], categories=orden_educ, ordered=True)

# Pivotear para el heatmap
heatmap_data = be_clean.pivot_table(
    index='sector', columns='nivel_educ', values='brecha_pct'
)

# Ordenar sectores por brecha promedio descendente
heatmap_data['_media'] = heatmap_data.mean(axis=1)
heatmap_data = heatmap_data.sort_values('_media', ascending=True).drop(columns='_media')

fig3 = px.imshow(
    heatmap_data,
    color_continuous_scale='RdBu_r',   # rojo = hombres ganan más, azul = mujeres
    color_continuous_midpoint=0,
    text_auto='.1f',
    labels={'color': 'Brecha (%)'},
    title='Brecha salarial por sector y nivel educativo · 2025<br><sup>% diferencia en ingreso mediano por hora. Rojo: hombres ganan más. Azul: mujeres ganan más.</sup>',
    aspect='auto',
)

fig3.update_layout(
    template='plotly_white',
    xaxis_title='Nivel educativo',
    yaxis_title='',
    font=dict(family='Arial, sans-serif', size=12),
    coloraxis_colorbar=dict(title='Brecha %', ticksuffix='%'),
    margin=dict(t=90, b=60, l=260, r=60),
    height=450,
)

fig3.write_html(f'{RUTA_DASH}/g3_heatmap_educ_sector.html')
fig3.show()
print('G3 guardada.')

/tmp/ipykernel_8355/4040895311.py:9: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



G3 guardada.


---
## G4 — Barras agrupadas: ingreso por hora + informalidad como contexto (2025)

Compara el ingreso mediano por hora de hombres y mujeres en cada sector.
La informalidad aparece como anotación en los sectores donde hay mayor
diferencia entre géneros, para contextualizar sin convertirla en el eje central.

In [7]:
# Preparar datos: ingreso por sector para los sectores con mayor presencia femenina
# (los que tienen brecha positiva, donde el análisis tiene más relevancia narrativa)
sectores_enfoque = [
    'Manufactura', 'Comercio', 'Restaurantes y hospedaje',
    'Servicios sociales', 'Servicios diversos',
    'Servicios financieros y profesionales'
]

bs25_focus = bs[bs['anio'] == 2025].copy()
bs25_focus = bs25_focus[bs25_focus['sector'].isin(sectores_enfoque)]

# Convertir a formato long para px.bar
bs_long = bs25_focus.melt(
    id_vars=['sector', 'brecha_pct'],
    value_vars=['Hombre', 'Mujer'],
    var_name='sexo',
    value_name='ingreso_hora'
).sort_values(['brecha_pct', 'sexo'], ascending=[False, True])

fig4 = px.bar(
    bs_long,
    x='sector', y='ingreso_hora', color='sexo',
    barmode='group',
    color_discrete_map=COLOR_MAPA,
    text='ingreso_hora',
    labels={'ingreso_hora': 'Ingreso mediano por hora (pesos)', 'sector': '', 'sexo': ''},
    title='Ingreso mediano por hora en sectores con mayor presencia femenina · 2025',
    category_orders={'sector': sectores_enfoque},
)

fig4.update_traces(
    texttemplate='$%{text:.0f}',
    textposition='outside',
    textfont=dict(size=10),
)

# Anotaciones de informalidad femenina en sectores clave
# Se muestra solo donde la diferencia entre géneros en informalidad es alta
inf_mujeres = inf[inf['sexo']=='Mujer'].set_index('sector')

for sector in ['Manufactura', 'Comercio', 'Restaurantes y hospedaje']:
    if sector in inf_mujeres.index:
        pct = inf_mujeres.loc[sector, 'pct_informalidad']
        # Posición y del ingreso de la mujer en ese sector
        ing_m = bs_long[(bs_long['sector']==sector) & (bs_long['sexo']=='Mujer')]['ingreso_hora'].values
        if len(ing_m) > 0:
            fig4.add_annotation(
                x=sector,
                y=ing_m[0] * 0.5,
                text=f'{pct:.0f}%<br>informal',
                showarrow=False,
                font=dict(size=9, color='gray'),
                align='center',
            )

# Nota explicativa de las anotaciones
fig4.add_annotation(
    xref='paper', yref='paper', x=0.01, y=-0.18,
    text='Nota: el porcentaje sobre las barras de mujeres indica la tasa de informalidad femenina en ese sector.',
    showarrow=False, font=dict(size=9, color='gray'), xanchor='left',
)

fig4.update_layout(
    template='plotly_white',
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Arial, sans-serif', size=12),
    margin=dict(t=60, b=90, l=60, r=40),
    height=480,
    yaxis=dict(range=[0, 115]),
)

fig4.write_html(f'{RUTA_DASH}/g4_ingreso_sector_informalidad.html')
fig4.show()
print('G4 guardada.')

G4 guardada.


In [8]:
archivos = os.listdir(RUTA_DASH)
print(f'Archivos en {RUTA_DASH}:')
for a in sorted(archivos):
    mb = os.path.getsize(f'{RUTA_DASH}/{a}') / 1024
    print(f'  {a}  ({mb:.0f} KB)')
print(f'\nSiguiente paso: tablero.ipynb')

Archivos en /content/drive/MyDrive/dashboard:
  g1_serie_nacional.html  (4462 KB)
  g2_brecha_sector.html  (4462 KB)
  g3_heatmap_educ_sector.html  (4461 KB)
  g4_ingreso_sector_informalidad.html  (4462 KB)

Siguiente paso: tablero.ipynb
